<a href="https://colab.research.google.com/github/Chris-p05/OptiHire/blob/main/OptiHireDataScraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- STEP 1: GLOBAL SETUP & LIBRARIES ---
!pip install torch tensorflow transformers nltk spacy pymupdf scrapingbee kaggle -q
import os, json, base64, requests, pymupdf, spacy, pandas as pd
from scrapingbee import ScrapingBeeClient
from google.colab import userdata

In [ ]:
# --- STEP 2: SECURE CREDENTIALS --- (SET THEM UP ON THE SECRETS TAB)
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Kaggle secrets loaded.")
except Exception:
    print("Kaggle secrets not found. Use manual dataset upload.")

In [ ]:
# --- STEP 3: DATA ACQUISITION (THE "GOOD" DATASET) ---
!kaggle datasets download -d snehaanbhawal/resume-dataset
!unzip -o resume-dataset.zip

In [ ]:
# --- STEP 4: PROFESSIONAL LOGIC ENGINE ---
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    spacy.cli.download("en_core_web_md")
    nlp = spacy.load("en_core_web_md")

def clean_and_anonymize(text):
    """
    Combines PII redaction and text normalization into one high-tier pass.

    """
    if not isinstance(text, str): return ""

    # 1. Anonymize to eliminate human bias
    doc = nlp(text)
    redacted_text = text
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "GPE", "DATE"]:
            redacted_text = redacted_text.replace(ent.text, f"[{ent.label_}_REDACTED]")

    # 2. Normalize for Semantic Analysis
    clean_doc = nlp(redacted_text.lower())
    tokens = [t.lemma_ for t in clean_doc if not t.is_stop and not t.is_punct]
    return " ".join(tokens)

def get_base64_resume(file_path):
    with open(file_path, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

In [ ]:
# --- STEP 5: BATCH PROCESSING (KAGGLE DATA) --- THIS IS WHERE WE TEST A SPECIFIC RESUME AGAINST THE DATASET
# We use 'Resume.csv' and 'Resume_str' from the new dataset.
try:
    df = pd.read_csv('Resume/Resume.csv') # Adjust path if unzip structure differs
except:
    df = pd.read_csv('Resume.csv')

processed_data = []
print("Processing the REAL resume text now!")

# Processing 50 samples to test the logic
for index, row in df.head(50).iterrows():
    processed_data.append({
        "id": row['ID'],
        "category": row['Category'],
        "clean_resume": clean_and_anonymize(row['Resume_str'])
    })

In [ ]:
# --- STEP 6: SCRAPINGBEE & INDIVIDUAL EVALUATION --- (AGAIN, SET THEM UP IN SECRETS TAB)
# Connects your live job data to a specific file upload.
job_data = {}
try:
    client = ScrapingBeeClient(api_key=userdata.get('SCRAPINGBEE_KEY'))

    response = client.get(
        'https://www.simplyhired.com/search?q=python+developer',
        params={'ai_query': 'Extract the job title as a JSON object with the key "job_title"', 'render_js': True}
    )

    # Debugging: Print status code and response text
    print(f"ScrapingBee Response Status Code: {response.status_code}")
    print(f"ScrapingBee Raw Response Text: {response.text}") # Print full response text

    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
    job_data = response.json()
except Exception as e:
    print(f"ScrapingBee Error: {e}. Check your API key and request parameters!")

# Evaluate "my_resume.pdf" if the user managed to upload it.
pdf_file = "my_resume.pdf"
if os.path.exists(pdf_file):
    encoded_resume = get_base64_resume(pdf_file)
    print(f"Ready to evaluate {pdf_file} against {job_data.get('job_title', 'Job Posting')}")
else:
    print(f"Warning: {pdf_file} not found. Upload it to the folder!")

print(f"\nPhase 1 & 2 COMPLETE. {len(processed_data)} resumes processed with zero bias.")

In [ ]:
try:
    scrapingbee_key = userdata.get('SCRAPINGBEE_KEY')
    if scrapingbee_key:
        print(f"SCRAPINGBEE_KEY is set (first 5 chars: {scrapingbee_key[:5]}*****)")
    else:
        print("SCRAPINGBEE_KEY is not found or is empty in Colab Secrets.")
except Exception as e:
    print(f"Error accessing SCRAPINGBEE_KEY from Colab Secrets: {e}")

In [ ]:
# --- STEP 7: SEMANTIC MATCHING ENGINE (WEEK 2 GOAL) ---
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Load the SBERT model (Optimized for performance)
# all-MiniLM-L6-v2 is fast and perfect for resume matching.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Move model to GPU if available (which we confirmed in step 1)
if torch.cuda.is_available():
    model = model.to('cuda')

def get_match_score(resume_text, job_description_text):
    """
    Calculates the semantic similarity between a resume and a job description.
    Returns a score from 0-100%.
    """
    # Encode both texts into semantic vectors (embeddings)
    resume_emb = model.encode(resume_text, convert_to_tensor=True)
    job_emb = model.encode(job_description_text, convert_to_tensor=True)

    # Compute Cosine Similarity (0 to 1)
    cosine_score = util.cos_sim(resume_emb, job_emb)

    # Convert to 0-100% scale as planned in our slides
    return round(float(cosine_score) * 100, 2)

# --- EXAMPLE TEST ---
# Testing our cleaned resumes against a sample job requirement
sample_jd = "Looking for Python developer with database experience"
clean_jd = clean_and_anonymize(sample_jd)

print(f"Testing against JD: '{sample_jd}'\n")
for resume in processed_data[:5]: # Testing the first 5 from our processed batch
    score = get_match_score(resume['clean_resume'], clean_jd)
    print(f"Resume ID {resume['id']} | Category: {resume['category']} | Match Score: {score}%")

In [ ]:
# --- STEP 8: MODEL REFINEMENT - KEYWORD CLUSTERING & WEIGHTING (WEEK 3) ---

def extract_keywords(text):
    """
    Uses spaCy to extract technical keywords and noun chunks.
    """
    doc = nlp(text.lower())
    # Extracting nouns and proper nouns as 'keywords'
    keywords = set([token.text for token in doc if token.pos_ in ["NOUN", "PROPN"] and not token.is_stop])
    return keywords

def refined_match_score(resume_text, job_description_text):
    # 1. Get Base Semantic Score (from our Step 7 model)
    semantic_score = get_match_score(resume_text, job_description_text)

    # 2. Calculate Keyword Alignment
    jd_keywords = extract_keywords(job_description_text)
    resume_keywords = extract_keywords(resume_text)

    if not jd_keywords: return semantic_score

    # Calculate how many JD keywords appear in the resume
    intersection = jd_keywords.intersection(resume_keywords)
    keyword_score = (len(intersection) / len(jd_keywords)) * 100

    # 3. Weighted Final Score (70% Semantic + 30% Keyword Match)
    final_score = (0.7 * semantic_score) + (0.3 * keyword_score)
    return round(final_score, 2)

# --- TEST THE REFINEMENT ---
print(f"Refining evaluation for the top candidate...")
top_resume = processed_data[0]['clean_resume']
test_jd = "Looking for a Python developer with experience in SQL, PyTorch, and NLP."

legacy_score = get_match_score(top_resume, test_jd)
new_score = refined_match_score(top_resume, test_jd)

print(f"Original SBERT Score: {legacy_score}%")
print(f"Refined Weighted Score: {new_score}%")

In [ ]:
# --- STEP 9: THE FINAL LEADERBOARD (THE "ACTION" PHASE) ---

def generate_leaderboard(job_description, resume_batch, top_n=10):
    results = []

    # Use the live scraped data if available, otherwise use sample
    target_jd = job_description if job_description else "Senior Python Developer PyTorch NLP"
    clean_jd = clean_and_anonymize(target_jd)

    print(f"Ranking candidates for: {target_jd[:50]}...")

    for resume in resume_batch:
        # Calculate the high-tier refined score
        score = refined_match_score(resume['clean_resume'], clean_jd)

        results.append({
            "Candidate ID": resume['id'],
            "Category": resume['category'],
            "Match Score": f"{score}%"
        })

    # Sort by score descending (highest first)
    leaderboard_df = pd.DataFrame(results)
    leaderboard_df['Match Score'] = leaderboard_df['Match Score'].str.replace('%', '').astype(float)
    leaderboard_df = leaderboard_df.sort_values(by="Match Score", ascending=False).reset_index(drop=True)

    return leaderboard_df.head(top_n)

# EXECUTE THE FINAL RANKING
if 'job_data' in locals() and job_data:
    # Use the 'body' or 'requirements' from ScrapingBee
    jd_text = job_data.get('requirements', job_data.get('body', 'Python Developer SQL'))
else:
    jd_text = "Experienced Data Scientist with Machine Learning and Python skills"

final_rankings = generate_leaderboard(jd_text, processed_data)

# Show the results in a way that looks good for the presentation
from google.colab import data_table
data_table.DataTable(final_rankings, include_index=False, num_rows_per_page=10)